# 06 — Distribution-Aware Hybrid

**Project**: From Clinical Jargon to Plain Language — Medical Text Simplification  
**Purpose**: Test the hypothesis from paper Sec V-D (Hybrid Regression). The naive hybrid (`04_hybrid`) fails because T5 was trained on **original** sources but at inference receives **rule-substituted** sources — distributional shift. This notebook removes the shift by applying the same rule-based substitution to **training and validation** sources before fine-tuning.

**Pipeline**:
- Train: `rules(train_src) → target` (targets unchanged)
- Inference: `rules(test_src) → T5 → prediction`
- Metric source = **original** test source (fair comparison vs. other systems)

**Output**: `predictions/hybrid_aware.jsonl`, `results/metrics.csv` (new `hybrid_aware` row)  
**Run**: Top-to-bottom on Colab T4. Restart kernel after install cell.

## 0. Colab Setup

In [ ]:
from google.colab import userdata, drive
import os, sys, shutil, subprocess

drive.mount('/drive')

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_URL = f'https://{GITHUB_TOKEN}@github.com/IbrahimHanafy2222/NLP-Project.git'
if not os.path.exists('/content/NLP-Project'):
    result = subprocess.run(['git', 'clone', GITHUB_URL], capture_output=True, text=True, cwd='/content')
    print(result.stdout or result.stderr)

project_root = '/content/NLP-Project'
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Working directory:', os.getcwd())

if not os.path.exists('data/processed'):
    shutil.copytree('/drive/MyDrive/NLP_Project/processed', 'data/processed')
    print('Loaded data/processed from Drive')
if not os.path.exists('data/medical_dict.json'):
    shutil.copy('/drive/MyDrive/NLP_Project/medical_dict.json', 'data/medical_dict.json')
    print('Loaded medical_dict.json from Drive')

## 1. Install

In [ ]:
import importlib.util as _ilu
if _ilu.find_spec('textstat') is None:
    get_ipython().system('pip install git+https://github.com/feralvam/easse.git sacrebleu "transformers>=4.35" datasets==2.18.0 torch spacy==3.7.4 pandas==2.2.1 textstat scikit-learn==1.4.1.post1')
    get_ipython().system('python -m spacy download en_core_web_sm')
    print('Packages installed. Runtime restarting — re-run from Config cell after restart.')
    import os; os.kill(os.getpid(), 9)
else:
    print('Packages already installed.')

## 2. Config

Same hyperparameters as `03_t5_finetune.ipynb` for **controlled comparison**. Only training inputs differ.

In [ ]:
MODEL_NAME       = "t5-small"
MODEL_SHORT_NAME = "hybrid_aware"

INPUT_PREFIX = "simplify: "

MAX_INPUT_LENGTH  = 256
MAX_TARGET_LENGTH = 128

BATCH_SIZE       = 8
GRAD_ACCUM_STEPS = 1
LEARNING_RATE    = 5e-4
NUM_EPOCHS       = 10
SEED             = 42

NUM_BEAMS = 4

DICT_PATH            = "data/medical_dict.json"
OUTPUT_DIR           = f"checkpoints/{MODEL_SHORT_NAME}"
DRIVE_CHECKPOINT_DIR = f"/drive/MyDrive/NLP_Project/checkpoints/{MODEL_SHORT_NAME}"
PREDICTIONS_FILE     = f"predictions/{MODEL_SHORT_NAME}.jsonl"

WANDB_ENABLED = False

print(f"MODEL_SHORT_NAME: {MODEL_SHORT_NAME}")
print(f"DICT_PATH:        {DICT_PATH}")
print(f"OUTPUT_DIR:       {OUTPUT_DIR}")

## 3. Imports & Seeds

In [ ]:
import random, json, os, sys, shutil
import numpy as np
import pandas as pd
import torch
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)
from datasets import load_from_disk

_COLAB_PROJECT = '/content/NLP-Project'
if os.path.exists(_COLAB_PROJECT):
    os.chdir(_COLAB_PROJECT)
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.metrics import compute_sari, compute_bleu, compute_fkgl
from src.rule_based import RuleBasedSimplifier

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROCESSED_DIR   = os.path.join(project_root, 'data/processed')
PREDICTIONS_DIR = os.path.join(project_root, 'predictions')
RESULTS_DIR     = os.path.join(project_root, 'results')

print('CUDA:', torch.cuda.is_available())

## 4. Load Data + Apply Rule-Based Substitution

Key step: apply `substitute_jargon` to **train and val sources only**. Targets unchanged. Test sources untouched here — substitution applied at inference time in Section 9.

We use `substitute_jargon` only (not full `simplify`) to match what the naive hybrid (`04_hybrid`) applies at inference — isolates the distributional-shift effect from sentence splitting / parenthetical removal.

In [ ]:
dataset = load_from_disk(PROCESSED_DIR)
train_dataset = dataset['train']
val_dataset   = dataset['val']
test_dataset  = dataset['test']

simplifier = RuleBasedSimplifier(dict_path=DICT_PATH)
print(f'Loaded simplifier with {len(simplifier.dictionary)} dictionary entries')

def _substitute_sources(batch):
    return {'source': [simplifier.substitute_jargon(s) for s in batch['source']],
            'target': batch['target']}

train_sub = train_dataset.map(_substitute_sources, batched=True, batch_size=512,
                              desc='Substituting train sources')
val_sub   = val_dataset.map(_substitute_sources,   batched=True, batch_size=512,
                            desc='Substituting val sources')

n_changed = sum(1 for a, b in zip(train_dataset['source'], train_sub['source']) if a != b)
print(f'Train: {len(train_sub):,} pairs, {n_changed:,} sources changed by substitution ({100*n_changed/len(train_sub):.1f}%)')
print(f'Val:   {len(val_sub):,} pairs')
print(f'Test:  {len(test_dataset):,} pairs (untouched here)')

print('\nExample substitution:')
for i in range(len(train_dataset)):
    if train_dataset[i]['source'] != train_sub[i]['source']:
        print('  ORIG:', train_dataset[i]['source'][:140])
        print('  SUB :', train_sub[i]['source'][:140])
        break

## 5. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    inputs = [INPUT_PREFIX + s for s in examples['source']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=examples['target'],
                       max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

orig_cols = train_sub.column_names
tokenized_train = train_sub.map(preprocess_function, batched=True, remove_columns=orig_cols)
tokenized_val   = val_sub.map(preprocess_function,   batched=True, remove_columns=orig_cols)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=None, padding=True)
print('Tokenized shapes — train:', tokenized_train.shape, 'val:', tokenized_val.shape)

## 6. Model

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to('cuda')
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params:,}')

## 7. Train

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=DRIVE_CHECKPOINT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    per_device_eval_batch_size=4,
    learning_rate=LEARNING_RATE,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    seed=SEED,
    fp16=True,
    report_to='wandb' if WANDB_ENABLED else 'none',
    logging_steps=50,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train(resume_from_checkpoint=True if os.path.isdir(DRIVE_CHECKPOINT_DIR) and any(d.startswith('checkpoint-') for d in os.listdir(DRIVE_CHECKPOINT_DIR)) else None)

print('Best eval_loss:', trainer.state.best_metric)
print('Best checkpoint:', trainer.state.best_model_checkpoint)

## 8. Resolve Best Checkpoint

In [ ]:
import glob as _glob

def _find_best_checkpoint(drive_dir):
    try:
        ckpt = trainer.state.best_model_checkpoint
        if ckpt and os.path.isdir(ckpt):
            return ckpt
    except NameError:
        pass
    state_file = os.path.join(drive_dir, 'trainer_state.json')
    if os.path.exists(state_file):
        state = json.load(open(state_file))
        ckpt = state.get('best_model_checkpoint', '')
        if ckpt and os.path.isdir(ckpt):
            return ckpt
        history = state.get('log_history', [])
        eval_entries = [e for e in history if 'eval_loss' in e]
        if eval_entries:
            best = min(eval_entries, key=lambda e: e['eval_loss'])
            ckpt = os.path.join(drive_dir, f'checkpoint-{int(best["step"])}')
            if os.path.isdir(ckpt):
                return ckpt
    ckpts = sorted(_glob.glob(os.path.join(drive_dir, 'checkpoint-*')),
                   key=lambda p: int(p.rsplit('-', 1)[-1]))
    return ckpts[-1] if ckpts else drive_dir

BEST_CHECKPOINT = _find_best_checkpoint(DRIVE_CHECKPOINT_DIR)
print('Loading from:', BEST_CHECKPOINT)

## 9. Inference — Substitute Test Sources, then Generate

**Critical**: feed substituted text to the model, but save the **original** source in the JSONL so SARI compares against the same input as every other system.

In [ ]:
infer_model = AutoModelForSeq2SeqLM.from_pretrained(BEST_CHECKPOINT).to('cuda')
infer_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
infer_model.eval()

INFER_BATCH = 16
rows = []

test_sources_orig = list(test_dataset['source'])
test_targets      = list(test_dataset['target'])
test_sources_sub  = [simplifier.substitute_jargon(s) for s in test_sources_orig]

for batch_start in range(0, len(test_sources_sub), INFER_BATCH):
    sub_batch  = test_sources_sub[batch_start:batch_start + INFER_BATCH]
    orig_batch = test_sources_orig[batch_start:batch_start + INFER_BATCH]
    ref_batch  = test_targets[batch_start:batch_start + INFER_BATCH]

    inputs = infer_tokenizer(
        [INPUT_PREFIX + s for s in sub_batch],
        return_tensors='pt', padding=True,
        max_length=MAX_INPUT_LENGTH, truncation=True,
    ).to('cuda')

    with torch.no_grad():
        out_ids = infer_model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_TARGET_LENGTH,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    preds = infer_tokenizer.batch_decode(out_ids, skip_special_tokens=True)
    for orig, pred, ref in zip(orig_batch, preds, ref_batch):
        rows.append({'source': orig, 'prediction': pred, 'reference': ref})

os.makedirs(PREDICTIONS_DIR, exist_ok=True)
with open(PREDICTIONS_FILE, 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

assert len(rows) == 1046, f'Expected 1046 rows, got {len(rows)}'
print(f'Saved {PREDICTIONS_FILE} ({len(rows)} rows)')
print('Sample prediction:', rows[0]['prediction'][:140])

## 10. Metrics

In [ ]:
loaded      = [json.loads(l) for l in open(PREDICTIONS_FILE)]
sources_lst = [r['source']     for r in loaded]
preds_lst   = [r['prediction'] for r in loaded]
refs_lst    = [r['reference']  for r in loaded]

sari        = compute_sari(sources_lst, preds_lst, refs_lst)
bleu        = compute_bleu(preds_lst, refs_lst)
fkgl_output = compute_fkgl(preds_lst)
fkgl_input  = compute_fkgl(sources_lst)
fkgl_delta  = fkgl_output - fkgl_input

metrics_dict = {
    'system':      MODEL_SHORT_NAME,
    'sari':        round(sari,        4),
    'bleu':        round(bleu,        4),
    'fkgl_input':  round(fkgl_input,  4),
    'fkgl_output': round(fkgl_output, 4),
    'fkgl_delta':  round(fkgl_delta,  4),
}
for k, v in metrics_dict.items():
    print(f'  {k:<12}: {v}')

## 11. Save to metrics.csv

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics_path = os.path.join(RESULTS_DIR, 'metrics.csv')

new_row = pd.DataFrame([metrics_dict])
if os.path.exists(metrics_path):
    existing = pd.read_csv(metrics_path)
    existing = existing[existing['system'] != MODEL_SHORT_NAME]
    combined = pd.concat([existing, new_row], ignore_index=True)
else:
    combined = new_row

combined.to_csv(metrics_path, index=False)
print(combined.to_string(index=False))

## 12. Comparison vs. Naive Hybrid + T5

In [ ]:
df = pd.read_csv(os.path.join(RESULTS_DIR, 'metrics.csv'))
order = ['rule_based', 't5_small', 'scifive', 'hybrid', 'hybrid_aware']
df_display = df.set_index('system').reindex([s for s in order if s in df['system'].values]).reset_index()
print('=' * 70)
print(df_display.to_string(index=False))
print('=' * 70)

def _get(name, col):
    rows = df[df['system'] == name]
    return rows.iloc[0][col] if len(rows) else None

t5    = _get('t5_small',     'sari')
naive = _get('hybrid',       'sari')
aware = _get('hybrid_aware', 'sari')

print(f'\nT5-small         SARI: {t5}')
print(f'Hybrid (naive)   SARI: {naive}  delta vs T5: {naive - t5:+.4f}')
print(f'Hybrid-aware     SARI: {aware}  delta vs T5: {aware - t5:+.4f}  delta vs naive: {aware - naive:+.4f}')

if aware > t5:
    print('\nResult: distribution-aware hybrid BEATS T5. Hypothesis confirmed — shift was the cause.')
elif aware > naive:
    print('\nResult: hybrid-aware beats naive hybrid but not T5. Shift partially explains regression; rules add no net value.')
else:
    print('\nResult: hybrid-aware did not improve. Rules actively harm generation regardless of train/test match.')

## 13. Assertions

In [ ]:
df = pd.read_csv(os.path.join(RESULTS_DIR, 'metrics.csv'))
assert MODEL_SHORT_NAME in df['system'].values
row = df[df['system'] == MODEL_SHORT_NAME].iloc[0]
assert row['fkgl_delta'] < 0, f'FKGL must decrease: {row["fkgl_delta"]}'
assert 0 <= row['sari'] <= 100
assert 0 <= row['bleu'] <= 100
print('All assertions passed.')